# Marketing Qualified Leads - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.olist_marketing_qualified_leads"
target_table = f"{catalog}.silver.olist_marketing_qualified_leads"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

root
 |-- mql_id: string (nullable = true)
 |-- first_contact_date: date (nullable = true)
 |-- landing_page_id: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(bronze_df.limit(10))

mql_id,first_contact_date,landing_page_id,origin,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
28bdfd5f057764b54c38770f95c69f2f,2018-01-14,22c29808c4f815213303f8933030604c,organic_search,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
126a0d10becbaafcb2e72ce6848cf32c,2018-05-15,6a110e795dd487f1cf8d7583671987af,email,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
f76136f54d14a3345951f25b7932366b,2018-05-24,d51b0d02f063ba1d053db6d97226eec3,email,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
2f838cade4a6012a6cb1016d1d8d95ed,2017-11-10,aeac92c0f5ae22a04ed3b746cce3a1b6,organic_search,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
7281942387a1a0c3f72a50a8b0bb0920,2017-12-25,88740e65d5d6b056e0cda098e1ea6313,social,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads


In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

Number of rows: 8000
Number of columns: 11


In [0]:
rescued_row_count = (
    bronze_df
    .filter(col("_rescued_data").isNotNull())
    .filter(trim(col("_rescued_data")) != "")
    .count()
)

print("Number of rescued rows:", rescued_row_count)

Number of rescued rows: 0


In [0]:
for column, dtype in bronze_df.dtypes[:4]:
    print(column)
    print("Null count:", bronze_df.filter(col(column).isNull()).count())
    print("Distinct count:", bronze_df.filter(col(column).isNotNull()).select(column).distinct().count())

    if dtype == "string":
        print("Extra whitespace row count:",
            (
                bronze_df.withColumn(f"{column}_trimmed", trim(col(column)))
                .filter(col(column) != col(f"{column}_trimmed"))
                .count()
            )
        )
    print("-"*20)

mql_id
Null count: 0
Distinct count: 8000
Extra whitespace row count: 0
--------------------
first_contact_date
Null count: 0
Distinct count: 336
--------------------
landing_page_id
Null count: 0
Distinct count: 495
Extra whitespace row count: 0
--------------------
origin
Null count: 60
Distinct count: 10
Extra whitespace row count: 0
--------------------


- mql_id is complete and unique, so it is the key.

## Transform to Silver

In [0]:
silver_df = (
    bronze_df
    .withColumnRenamed(
        "mql_id",
        "marketing_qualified_lead_id"
    )
    .withColumnRenamed(
        "origin",
        "lead_origin"
    )
)

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

root
 |-- marketing_qualified_lead_id: string (nullable = true)
 |-- first_contact_date: date (nullable = true)
 |-- landing_page_id: string (nullable = true)
 |-- lead_origin: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(silver_table_df.limit(10))

marketing_qualified_lead_id,first_contact_date,landing_page_id,lead_origin,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
28bdfd5f057764b54c38770f95c69f2f,2018-01-14,22c29808c4f815213303f8933030604c,organic_search,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
126a0d10becbaafcb2e72ce6848cf32c,2018-05-15,6a110e795dd487f1cf8d7583671987af,email,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
f76136f54d14a3345951f25b7932366b,2018-05-24,d51b0d02f063ba1d053db6d97226eec3,email,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
2f838cade4a6012a6cb1016d1d8d95ed,2017-11-10,aeac92c0f5ae22a04ed3b746cce3a1b6,organic_search,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
7281942387a1a0c3f72a50a8b0bb0920,2017-12-25,88740e65d5d6b056e0cda098e1ea6313,social,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads


In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())

Bronze row count: 8000
Silver row count: 8000
